In [ ]:
# ==========================================
# 11 - SHAP ANALYSIS V1
# CHỌN VÀ HUẤN LUYỆN CATBOOST
# ==========================================

import sys
import json
import joblib
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

from catboost import CatBoostClassifier
from sklearn.multioutput import MultiOutputClassifier

RANDOM_SEED = 42
lib.np.random.seed(RANDOM_SEED)

# Thư mục kết quả
output_dir = (
    ROOT_DIR
    / "experiments/v1_baseline/shap_analysis/results"
)

figure_dir = output_dir / "figures"
table_dir = output_dir / "tables"
model_dir = output_dir / "models"

for directory in [figure_dir, table_dir, model_dir]:
    directory.mkdir(parents=True, exist_ok=True)

# Đọc cấu hình CatBoost Baseline đã lưu
config_path = (
    ROOT_DIR
    / "experiments/v1_baseline/catboost/results/cat_config.json"
)

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

assert config["model"] == "CatBoost"

# Đọc Development
df_dev = lib.pd.read_csv(
    ROOT_DIR / "data/data-splits/development.csv",
    encoding="utf-8-sig"
)

id_col = "ma_hoc_sinh"

score_cols = config["feature_columns"]
choice_cols = config["target_columns"]

X = df_dev[score_cols].copy()
y = df_dev[choice_cols].copy()

assert X.shape == (9000, 96)
assert y.shape == (9000, 9)
assert X.notna().all().all()
assert y.isin([0, 1]).all().all()

# Khởi tạo lại cấu hình CatBoost V1
base_model = CatBoostClassifier(
    **config["hyperparameters"]
)

model = MultiOutputClassifier(
    estimator=base_model,
    n_jobs=1
)

# Huấn luyện trên Development
model.fit(X, y)

assert len(model.estimators_) == 9

print("Mô hình giải thích: CatBoost")
print("Số học sinh Development:", len(X))
print("Số đặc trưng:", X.shape[1])
print("Số nhãn:", y.shape[1])
print("Random Seed:", RANDOM_SEED)
print("PASS: CatBoost đã được huấn luyện trên Development.")

In [ ]:
# ==========================================
# TÍNH SHAP VALUES CHO 9 NHÃN
# ==========================================

from catboost import Pool

# Lấy mẫu giải thích cố định
SHAP_SAMPLE_SIZE = 300

X_explain = X.sample(
    n=SHAP_SAMPLE_SIZE,
    random_state=RANDOM_SEED
).copy()

explain_ids = df_dev.loc[
    X_explain.index, id_col
].reset_index(drop=True)

shap_values = {}
shap_base_values = {}

for i, label in enumerate(choice_cols):

    estimator = model.estimators_[i]

    pool = Pool(
        X_explain,
        feature_names=score_cols
    )

    # Cột cuối là expected value
    shap_result = estimator.get_feature_importance(
        data=pool,
        type="ShapValues"
    )

    values = shap_result[:, :-1]
    base_values = shap_result[:, -1]

    assert values.shape == (SHAP_SAMPLE_SIZE, 96)
    assert lib.np.isfinite(values).all()

    shap_values[label] = values
    shap_base_values[label] = base_values

    print(
        f"{i+1}/9 - {label}: "
        f"SHAP shape = {values.shape}"
    )

print("\nPASS: Đã tính SHAP Values cho cả 9 nhãn.")

In [ ]:
# ==========================================
# SHAP FEATURE IMPORTANCE
# ==========================================

importance_records = []

for label in choice_cols:

    values = shap_values[label]

    mean_abs_shap = lib.np.abs(values).mean(axis=0)

    for feature, importance in zip(
        score_cols, mean_abs_shap
    ):
        importance_records.append({
            "Nhãn": label,
            "Đặc trưng": feature,
            "Mean_Abs_SHAP": importance
        })

shap_importance = lib.pd.DataFrame(
    importance_records
)

shap_importance = shap_importance.sort_values(
    ["Nhãn", "Mean_Abs_SHAP"],
    ascending=[True, False]
)

# Top 10 đặc trưng cho từng nhãn
top10_features = (
    shap_importance
    .groupby("Nhãn", sort=False)
    .head(10)
    .reset_index(drop=True)
)

for label in choice_cols:

    print(f"\n{'=' * 60}")
    print(f"TOP 10 ĐẶC TRƯNG - {label}")
    print("=" * 60)

    display(
        top10_features[
            top10_features["Nhãn"] == label
        ][["Đặc trưng", "Mean_Abs_SHAP"]]
    )

In [ ]:
# ==========================================
# SHAP SUMMARY PLOT - 9 NHÃN
# ==========================================

import shap

for i, label in enumerate(choice_cols):

    lib.plt.figure(figsize=(10, 7))

    shap.summary_plot(
        shap_values[label],
        X_explain,
        feature_names=score_cols,
        plot_type="bar",
        max_display=15,
        show=False
    )

    lib.plt.title(
        f"SHAP Summary - {label}",
        fontsize=13
    )

    lib.plt.tight_layout()

    filename = f"shap_summary_label_{i+1}.png"

    lib.plt.savefig(
        figure_dir / filename,
        dpi=200,
        bbox_inches="tight"
    )

    lib.plt.show()
    lib.plt.close()

    print(f"Đã lưu: {filename}")

In [ ]:
# ==========================================
# SHAP BEESWARM PLOT - 9 NHÃN
# ==========================================

for i, label in enumerate(choice_cols):

    explanation = shap.Explanation(
        values=shap_values[label],
        base_values=shap_base_values[label],
        data=X_explain.to_numpy(),
        feature_names=score_cols
    )

    shap.plots.beeswarm(
        explanation,
        max_display=15,
        show=False
    )

    fig = lib.plt.gcf()
    fig.set_size_inches(11, 8)

    lib.plt.title(
        f"SHAP Beeswarm - {label}",
        fontsize=13
    )

    filename = f"shap_beeswarm_label_{i+1}.png"

    fig.savefig(
        figure_dir / filename,
        dpi=200,
        bbox_inches="tight"
    )

    lib.plt.show()
    lib.plt.close(fig)

    print(f"Đã lưu: {filename}")

In [ ]:
# ==========================================
# SHAP DEPENDENCE PLOT
# ==========================================

dependence_records = []

for i, label in enumerate(choice_cols):

    # Chọn đặc trưng quan trọng nhất của nhãn
    top_feature = (
        shap_importance[
            shap_importance["Nhãn"] == label
        ]
        .iloc[0]["Đặc trưng"]
    )

    dependence_records.append({
        "Nhãn": label,
        "Đặc trưng phân tích": top_feature
    })

    lib.plt.figure(figsize=(9, 6))

    shap.dependence_plot(
        top_feature,
        shap_values[label],
        X_explain,
        feature_names=score_cols,
        interaction_index=None,
        show=False
    )

    lib.plt.title(
        f"SHAP Dependence - {label}",
        fontsize=12
    )

    lib.plt.tight_layout()

    filename = f"shap_dependence_label_{i+1}.png"

    lib.plt.savefig(
        figure_dir / filename,
        dpi=200,
        bbox_inches="tight"
    )

    lib.plt.show()
    lib.plt.close()

    print(f"{label}: {top_feature}")


dependence_df = lib.pd.DataFrame(
    dependence_records
)

display(dependence_df)

In [ ]:
# ==========================================
# PHÂN TÍCH SHAP THEO KHỐI LỚP
# ==========================================

grade_records = []

for label in choice_cols:

    label_importance = shap_importance[
        shap_importance["Nhãn"] == label
    ]

    total_importance = (
        label_importance["Mean_Abs_SHAP"].sum()
    )

    for grade in [6, 7, 8, 9]:

        grade_features = label_importance[
            label_importance["Đặc trưng"].str.startswith(
                f"L{grade}_"
            )
        ]

        grade_importance = (
            grade_features["Mean_Abs_SHAP"].sum()
        )

        grade_records.append({
            "Nhãn": label,
            "Lớp": f"L{grade}",
            "Số đặc trưng": len(grade_features),
            "Total_Abs_SHAP": grade_importance,
            "Contribution_%": (
                grade_importance / total_importance * 100
                if total_importance > 0 else 0
            )
        })

grade_shap_df = lib.pd.DataFrame(grade_records)

grade_pivot = grade_shap_df.pivot(
    index="Nhãn",
    columns="Lớp",
    values="Contribution_%"
)[["L6", "L7", "L8", "L9"]]

print("=" * 60)
print("ĐÓNG GÓP SHAP THEO KHỐI LỚP (%)")
print("=" * 60)

display(grade_pivot.round(2))

# Kiểm tra số đặc trưng
assert grade_shap_df["Số đặc trưng"].eq(24).all()

# Vẽ biểu đồ đóng góp theo lớp
ax = grade_pivot.plot(
    kind="bar",
    stacked=True,
    figsize=(13, 7)
)

ax.set_title(
    "SHAP Contribution by Grade - CatBoost V1",
    fontsize=14
)

ax.set_xlabel("Target Label")
ax.set_ylabel("Relative Mean |SHAP| (%)")
ax.legend(title="Grade")

lib.plt.xticks(rotation=35, ha="right")
lib.plt.tight_layout()

lib.plt.savefig(
    figure_dir / "shap_contribution_by_grade.png",
    dpi=200,
    bbox_inches="tight"
)

lib.plt.show()

# Trung bình tỷ lệ đóng góp trên 9 nhãn
grade_overall = (
    grade_shap_df
    .groupby("Lớp")["Contribution_%"]
    .mean()
    .reset_index(name="Mean_Contribution_%")
)

print("\nTRUNG BÌNH ĐÓNG GÓP THEO LỚP")
display(grade_overall.round(2))

In [ ]:
# ==========================================
# SAVE SHAP ANALYSIS V1
# ==========================================

# 1. Lưu các bảng kết quả
reports = {
    "shap_feature_importance.csv": shap_importance,
    "shap_top10_features.csv": top10_features,
    "shap_dependence_features.csv": dependence_df,
    "shap_contribution_by_grade.csv": grade_shap_df,
    "shap_grade_summary.csv": grade_overall
}

for filename, data in reports.items():

    data.to_csv(
        table_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Đã lưu: {filename}")


# 2. Lưu SHAP Values của 9 nhãn
shap_arrays = {
    f"label_{i+1}": shap_values[label]
    for i, label in enumerate(choice_cols)
}

lib.np.savez_compressed(
    table_dir / "shap_values_9labels.npz",
    **shap_arrays
)

# 3. Lưu danh sách học sinh dùng để giải thích
explain_ids.to_frame(name=id_col).to_csv(
    table_dir / "shap_sample_ids.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. Lưu mô hình CatBoost huấn luyện trên Development
joblib.dump(
    model,
    model_dir / "catboost_v1_shap_model.joblib"
)

# 5. Lưu metadata
metadata = {
    "experiment": "SHAP Analysis V1",
    "model": "CatBoost",
    "wrapper": "MultiOutputClassifier",
    "random_seed": RANDOM_SEED,
    "training_source": "Development only",
    "training_samples": len(X),
    "explanation_samples": len(X_explain),
    "n_features": len(score_cols),
    "n_labels": len(choice_cols),
    "shap_output": "Raw score (log-odds)",
    "independent_test_used": False,
    "target_columns": choice_cols,
    "feature_columns": score_cols,
    "hyperparameters": config["hyperparameters"]
}

with open(
    output_dir / "shap_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=4
    )

# 6. Kiểm tra kết quả
print("=" * 60)
print("SHAP ANALYSIS V1 - SAVED RESULTS")
print("=" * 60)

print("Số bảng kết quả:", len(list(table_dir.glob("*"))))
print("Số biểu đồ:", len(list(figure_dir.glob("*.png"))))
print("Số mô hình:", len(list(model_dir.glob("*.joblib"))))

assert len(list(figure_dir.glob("*.png"))) == 28

print("\nPASS: Đã lưu kết quả SHAP Analysis V1.")
print("Thư mục:", output_dir.resolve())